# Soluzioni — capitolo 8, esercizio 1: dimensione del latente contro PCA

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import torch, numpy as np
from torch import nn
from torchvision import datasets, transforms
tr = transforms.ToTensor(); mtr = datasets.MNIST("../data", train=True, download=True, transform=tr); mte = datasets.MNIST("../data", train=False, download=True, transform=tr)
X_tr = mtr.data.float().div(255).view(-1, 784); X_te = mte.data.float().div(255).view(-1, 784)
class AE(nn.Module):
    def __init__(s, d):
        super().__init__(); s.enc = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, d)); s.dec = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid())
    def forward(s, x): z = s.enc(x); return s.dec(z), z
mu = X_tr.mean(0); U, S, Vt = torch.linalg.svd(X_tr[:20000] - mu, full_matrices=False)
for d in (4, 8, 16, 32, 64):
    fissa_seme(42); m = AE(d); opt = torch.optim.Adam(m.parameters(), lr=1e-3); fn = nn.MSELoss()
    for e in range(6):
        perm = torch.randperm(60000)
        for i in range(0, 60000, 128): xb = X_tr[perm[i:i + 128]]; opt.zero_grad(); fn(m(xb)[0], xb).backward(); opt.step()
    m.eval()
    with torch.no_grad(): e_ae = ((m(X_te)[0] - X_te) ** 2).mean().item()
    Z = (X_te - mu) @ Vt[:d].T; e_p = ((Z @ Vt[:d] + mu - X_te) ** 2).mean().item()
    print(f"d = {d:2d}: autoencoder {e_ae:.4f}   PCA {e_p:.4f}")